In [32]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

In [33]:
df = pd.read_excel("Concrete_Data.xls")

X = df.iloc[:, :-1]   # all columns except the last one
y = df.iloc[:, -1]    # the last column (strength)


def categorize_strength(value):
    if value < 25.01:
        return 0  # low
    elif value < 55.02:
        return 1  # medium
    else:
        return 2  # high

y_classes = np.array([categorize_strength(val) for val in y])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_classes, test_size=0.2, random_state=42, stratify=y_classes
)

scaler = StandardScaler()
X_train_scaled_withoutCV = scaler.fit_transform(X_train)
X_test_scaled_withoutCV = scaler.transform(X_test)

# 1. Baseline Model

The baseline will be a model which compute the largest class on the training data, and predict everything in the
test-data as belonging to that class (corresponding to the optimal prediction by a logistic
regression model with a bias term and no features

In [34]:
num_low = np.sum(y_classes == 0)
num_medium = np.sum(y_classes == 1)
num_high = np.sum(y_classes == 2)

print(f"Class distribution:\n Low: {num_low}\n Medium: {num_medium}\n High: {num_high}")

Class distribution:
 Low: 295
 Medium: 588
 High: 147


In [35]:
# sicne the medium class has the most samples, the baseline model will predict this class everytime 

from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_scaled_withoutCV, y_train)
y_pred_base = baseline.predict(X_test_scaled_withoutCV)

print(confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))   

[[  0  59   0]
 [  0 118   0]
 [  0  29   0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        59
           1       0.57      1.00      0.73       118
           2       0.00      0.00      0.00        29

    accuracy                           0.57       206
   macro avg       0.19      0.33      0.24       206
weighted avg       0.33      0.57      0.42       206



/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [36]:
print(y_pred_base)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


# 2. ANN

In [37]:
from sklearn.neural_network import MLPClassifier

model_mlp = MLPClassifier(hidden_layer_sizes=(64, 32, 16), 
                      activation='relu', solver='adam',
                      #learning_rate='adaptive',
                      #learning_rate_init=0.005,
                      max_iter=3000,
                      random_state=42)


model_mlp.fit(X_train_scaled_withoutCV, y_train)
predictions_ann = model_mlp.predict(X_test_scaled_withoutCV)

test with normal train/test split

In [38]:
print(confusion_matrix(y_test, predictions_ann))
print(classification_report(y_test, predictions_ann))   

[[ 51   8   0]
 [  8 108   2]
 [  0   4  25]]
              precision    recall  f1-score   support

           0       0.86      0.86      0.86        59
           1       0.90      0.92      0.91       118
           2       0.93      0.86      0.89        29

    accuracy                           0.89       206
   macro avg       0.90      0.88      0.89       206
weighted avg       0.89      0.89      0.89       206



two level cross validation: 

In [39]:
print(type(y))

<class 'pandas.core.series.Series'>


We stored the results in a df, so there is no need to perform the two-fold CV everytime 

In [57]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

results = []
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# logistic regression model
logreg = LogisticRegression(max_iter=500)
logreg_params = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

#grid search: parameters than can be chosen
mlp_params = {
    'hidden_layer_sizes': [(32,), (64,), (128,), (64, 32), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],  
    'learning_rate_init': [0.001, 0.01],
    'learning_rate': ['constant', 'adaptive']
}


for fold_idx, (train_idx, test_idx)in enumerate(outer_cv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_classes[train_idx], y_classes[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Baseline
    baseline.fit(X_train_scaled, y_train)
    y_pred_base = baseline.predict(X_test_scaled)
    err_base = 1 - accuracy_score(y_test, y_pred_base)

    # logistic regression
    grid_logreg = GridSearchCV(logreg, logreg_params, cv=inner_cv, n_jobs=-1)
    grid_logreg.fit(X_train_scaled, y_train)
    best_logreg_params = grid_logreg.best_params_

    best_logreg = LogisticRegression(max_iter=500, C=best_logreg_params['C'])
    best_logreg.fit(X_train_scaled, y_train)
    y_pred_logreg = best_logreg.predict(X_test_scaled)
    err_logreg = 1 - accuracy_score(y_test, y_pred_logreg)


    # MLP
    grid_mlp = GridSearchCV(model_mlp, mlp_params, cv=inner_cv, n_jobs=-1)
    grid_mlp.fit(X_train_scaled, y_train)
    best_mlp_params = grid_mlp.best_params_
    
    #use best paramters for whole lopp
    best_mlp_model = MLPClassifier(
        hidden_layer_sizes=best_mlp_params['hidden_layer_sizes'],
        alpha=best_mlp_params['alpha'],
        activation=best_mlp_params['activation'],
        learning_rate=best_mlp_params['learning_rate'],
        learning_rate_init=best_mlp_params['learning_rate_init'],
        max_iter=500
    )

    best_mlp_model.fit(X_train_scaled, y_train)
    y_pred_mlp = best_mlp_model.predict(X_test_scaled)
    err_mlp = 1 - accuracy_score(y_test, y_pred_mlp)

    # results
    results.append({
        'Fold': fold_idx,
        'Baseline Error': err_base,
        'LogReg Error': err_logreg,
        'LogReg Params': grid_logreg.best_params_,
        'MLP Error': err_mlp,
        'MLP Params': grid_mlp.best_params_,
    })



/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Task 3: table for every fold

In [69]:
'''
results_df = pd.DataFrame(results)
mlp_params_df = pd.DataFrame(results_df['MLP Params'].tolist())
results_df = pd.concat([results_df, mlp_params_df], axis=1)


logreg_params_df = pd.DataFrame(results_df['LogReg Params'].tolist())
results_df = pd.concat([results_df, logreg_params_df], axis=1)

results_df = results_df[['Fold', 'hidden_layer_sizes', 'alpha', 'MLP Error', 'C', 'LogReg Error', 'Baseline Error']]
results_df.rename(columns={
    'hidden_layer_sizes':'h*', 
    'alpha':'λ*',
    'C':'C*'
}, inplace=True)
'''
#results_df.to_csv("results_classification_twofoldCV.csv", index=False)  

results_df = pd.read_csv("results_classification_twofoldCV.csv")
print(results_df)

   Fold         h*      λ*  MLP Error   C*  LogReg Error  Baseline Error
0     1      (64,)  0.0100   0.126214  100      0.189320        0.402913
1     2   (64, 32)  0.0001   0.116505    1      0.218447        0.504854
2     3   (64, 32)  0.0100   0.116505   10      0.203883        0.412621
3     4  (128, 64)  0.0100   0.169903   10      0.223301        0.432039
4     5  (128, 64)  0.0010   0.150485   10      0.203883        0.393204


task4: final parameters to calculate error 

In [70]:
#fpr MLP
mlp_params_df = results_df[['h*', 'λ*']]
most_common_mlp = mlp_params_df.value_counts().idxmax()
print("Most frequent MLP configuration:", most_common_mlp)


#for logistic regression
logreg_params = results_df['C*']
most_common_logreg = logreg_params.mode()[0]
print("Most frequent LogReg C:", most_common_logreg)


Most frequent MLP configuration: ('(128, 64)', np.float64(0.001))
Most frequent LogReg C: 10


In [71]:
numeric_results = results_df[['Baseline Error', 'LogReg Error', 'MLP Error']].copy()

# Berechne Mean ± SD
summary = numeric_results.agg(['mean', 'std'])
summary_row = summary.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}")
print(summary_row)

Baseline Error    0.429 ± 0.045
LogReg Error      0.208 ± 0.013
MLP Error         0.136 ± 0.024
dtype: object
